# Assignment 2: NLP Basics
In this assignment, you will implement and explore three fundamental concepts in Natural Language Processing: word embeddings (GloVe), subword tokenization (BPE), and recurrent neural networks for text generation (LSTM). This will give you a hands-on understanding of how text is represented and processed in modern NLP models.

**Due date: 2025.11.16 , 23:59**

### Environment Setup and Data Download
Before you begin, please make sure you have the necessary libraries installed. You can typically install them using pip.
```bash
pip install torch numpy requests

## Section 1: GloVe Embeddings
In this section, you will load the traditional GloVe embeddings, and explore the basic properties of the embedding space.

In [1]:
# Download the GloVe processed_processed_text files
# !wget http://nlp.stanford.edu/data/glove.6B.zip
# !unzip glove.6B.zip

### 1.1 Load the GloVe embeddings [5 pts]

In [2]:
import numpy as np

# Load GloVe word vectors from a text file into a dictionary
def load_glove_embeddings(glove_file_path):
    """
    Loads GloVe embeddings into a dictionary mapping words to their vector representations.
    """
    embeddings_dict = {}
    # TODO: Open the file and parse each line.
    # Each line contains a word followed by its vector values.
    # Convert the vector values to a numpy array.
    with open(glove_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.array(values[1:], dtype='float64')
            embeddings_dict[word] = vector

    return embeddings_dict

glove_file_path = 'glove.6B.100d.txt'  # Replace with the path to your GloVe file
glove_embeddings = load_glove_embeddings(glove_file_path)

In [3]:
# glove_embeddings['multimodal']

### 1.2 Find closest words [5 pts]
For a given word, you should find the most similar words in the vocabulary based on cosine similarity and output them along with their similarity scores.

In [4]:
# Function to find the closest words and their corresponding similarity values
def find_closest_words(word_vec, embeddings_dict, top_n=5):
    """
    Finds the top_n closest words to a given vector based on cosine similarity.
    Note: The input is a vector, not a word. This makes the function more versatile.
    """
    # TODO:
    # 1. Calculate the cosine similarity between the input word_vec and all other word vectors in embeddings_dict.
    # 2. Sort the words based on similarity in descending order.
    # 3. Return the top_n words and their similarity scores.
    similarities = {}
    for word, vec in embeddings_dict.items():
        if np.array_equal(word_vec, vec):
            continue  # skip the same word
        cosine_similarity = np.dot(word_vec, vec) / (np.linalg.norm(word_vec) * np.linalg.norm(vec))
        similarities[word] = cosine_similarity
    sorted_words = sorted(similarities.items(), key=lambda item: item[1], reverse=True)
    return sorted_words[:top_n]

chosen_word = 'man'
if chosen_word in glove_embeddings:
    chosen_word_vec = glove_embeddings[chosen_word]
    closest_words = find_closest_words(chosen_word_vec, glove_embeddings, top_n=5)
    print(f"The words closest to '{chosen_word}' are:")
    for word, similarity in closest_words:
        print(f"{word} with similarity of {similarity:.4f}")

The words closest to 'man' are:
woman with similarity of 0.8323
boy with similarity of 0.7915
one with similarity of 0.7789
person with similarity of 0.7527
another with similarity of 0.7522


### 1.3 Find new analogies [5 pts]
In the lecture, we discussed how linear relationships exist in the embedding space (e.g. king - man + woman ≈ queen). Please demonstrate a new analogy that was not mentioned in the lecture. You should perform the vector arithmetic like `vec(word1) - vec(word2) + vec(word3)` and find the word closest to the resulting vector.

In [5]:
# E.g., test cat - tiger == ? - wolf
# Assume ? to be something like 'dog'
word1 = 'cat'
word2 = 'tiger'
word3 = 'wolf'

word_vec = glove_embeddings[word1] - glove_embeddings[word2] + glove_embeddings[word3]
closest_words = find_closest_words(word_vec, glove_embeddings, top_n=5)
print(f"The words closest to the analogy '{word1} - {word2} + {word3}' are:")
for word, similarity in closest_words:
    print(f"{word} with similarity of {similarity:.4f}")

The words closest to the analogy 'cat - tiger + wolf' are:
dog with similarity of 0.6663
wolf with similarity of 0.6562
cat with similarity of 0.6357
puppy with similarity of 0.5529
dogs with similarity of 0.5469


## Section 2: BPE tokenizer

Byte Pair Encoding(BPE) is a subword tokenization technique that iteratively merges the most frequent adjacent byte pairs into subword units, creating a vocabulary that balances character-level granularity and whole-word tokens. This method is widely used in modern natural language processing to handle out-of-vocabulary words and optimize tokenization efficiency.

Let's look at an example. Given a sample string "banana bandana", we can calculate the frequency of the character pairs: 
```python
('a', 'n'): 4, ('n', 'a'): 3, ('b', 'a'): 2, ('a', ' '): 1, (' ','b'): 1, ('n', 'd'): 1, ('d', 'a'): 1
```
Which means that we can combine 'an' into a new token. In the next round, 'an' can now participate in the frequency count, giving:
```python
('b', 'an'): 2, ('an', 'a'): 2, ('an', 'an'): 1, ('a', ' '): 1, (' ','b'): 1, ('an', 'd'): 1, ('d', 'an'): 1
```
So we may get 'ban' as a new token. Similarly, 'ana' would be the most frequent pair in the next round. With three merges, we've added 'an', 'ban' and 'ana' into our vocabulary, and our string can now be converted to the following tokens:
```python
'ban', 'ana', ' ', 'ban','d' ,'ana'
```
So now we can use 6 tokens to represent the 14 characters.

You may wonder how this is better than word-level tokenization. First of all, it is more robust in out-of-vocabulary scenarios. For example, though the word "bandana" does exist in the GloVe embedding (look it up if you're not convinced), something like "banada" does not. When using GloVe embeddings, encountering "banada" during training would result in the default \<UNK\> token. In contrast, a BPE tokenizer can still infer the word's meaning through its sub-word tokens. Secondly, sub-word tokens include prefixes and suffixes that allow the model to learn different variations of a single word more efficiently.

In this section, you are required to implement a BPE tokenizer, and use one of the provided corpora to train it. You may train it on character level (starting with a vocabulary of all characters in the corpus) or byte level (starting with a vocabulary of all 256 possible byte values). You should verify that encoding and then decoding a sentence produces the original sentence. You may refer to (but not copy) the following implementations:
1. The tiktoken library: https://github.com/openai/tiktoken/blob/main/tiktoken/_educational.py
2. Kaparthy's minbpe repository: https://github.com/karpathy/minbpe

### 2.1 Implementation and Verification [15 pts]

In [6]:
import collections
class BPETokenizer:
    def __init__(self):
        self.vocab = {}
        self.merges = {}

    def train(self, text: str, vocab_size: int):
        """
        Trains the BPE tokenizer. You will need to store the learned merge rules and the final vocabulary.
        """
        # TODO:
        # 1. Initialize vocabulary with all unique characters in the text.
        word_counts = collections.Counter(text.split())
        tokenized_words = {}
        for word in word_counts.keys():
            tokenized_words[word] = list(word) + ['</w>']  # add end-of-word token
        unique_chars = sorted(list(set([char for word in tokenized_words.values() for char in word])))
        self.vocab = {char: i + 1 for i, char in enumerate(unique_chars)}
        self.vocab['<UNK>'] = 0  # add unknown token
        initial_vocab_size = len(self.vocab)
        
        # 2. Calculate the number of merges needed (vocab_size - initial_vocab_size).
        num_merges = vocab_size - initial_vocab_size

        # 3. Loop for the required number of merges:
        for rank in range(num_merges):
            # a. Find the most frequent adjacent pair of tokens in the text.
            pair_count = {}
            for word, count in word_counts.items():
                tokens = tokenized_words[word]  # type: ignore
                for idx in range(len(tokens) - 1):
                    pair = (tokens[idx], tokens[idx + 1])
                    if pair in pair_count:
                        pair_count[pair] += count
                    else:
                        pair_count[pair] = count
            if not pair_count:
                break
            most_frequent_pair = max(pair_count.items(), key=lambda item: item[1])[0]

            # b. Create a new token by merging this pair.
            new_token = ''.join(most_frequent_pair)

            # c. Add the new token to the vocabulary and record the merge rule.
            if new_token not in self.vocab:
                self.vocab[new_token] = len(self.vocab)
                self.merges[most_frequent_pair] = (rank, new_token)

            # d. Replace all occurrences of the pair in the text with the new token.
            for word in tokenized_words.keys():
                tokens = tokenized_words[word]
                new_tokens = []
                idx = 0
                while idx < len(tokens):
                    if idx < len(tokens) - 1 and (tokens[idx], tokens[idx + 1]) == most_frequent_pair:
                        new_tokens.append(new_token)
                        idx += 2
                    else:
                        new_tokens.append(tokens[idx])
                        idx += 1
                tokenized_words[word] = new_tokens
            
        # build reverse vocab for decoding
        self.reverse_vocab = {idx: token for token, idx in self.vocab.items()}

    def encode(self, text: str) -> list[int]:
        """
        Encodes a string into a list of token indices using the learned merge rules.
        """
        # TODO
        split_text = text.split()
        encoded_tokens = []
        for word in split_text:
            tokens = list(word) + ['</w>']
            while True:
                best_merge = (float('inf'), -1, "") # (rank, index, new_token)
                for i in range(len(tokens) - 1):
                    pair = (tokens[i], tokens[i+1])
                    if pair in self.merges:
                        rank, new_token = self.merges[pair]
                        if rank < best_merge[0]:
                            best_merge = (rank, i, new_token)
                if best_merge[1] != -1:
                    _, merge_index, new_token = best_merge
                    new_tokens = []
                    idx = 0
                    while idx < len(tokens):
                        if idx == merge_index:
                            new_tokens.append(new_token)
                            idx += 2
                        else:
                            new_tokens.append(tokens[idx])
                            idx += 1
                    tokens = new_tokens
                else:
                    break
            # add token of this word to encoded_tokens
            for token in tokens:
                encoded_tokens.append(self.vocab.get(token, self.vocab['<UNK>']))

        return encoded_tokens

    def decode(self, tokens: list[int]) -> str:
        """
        Decodes a list of token indices back into a text string.
        """
        # TODO
        decoded_tokens = [self.reverse_vocab[token] for token in tokens]
        # join decoded tokens and replace '</w>' with space
        decoded_string = ''.join(decoded_tokens).replace('</w>', ' ').strip()
        return decoded_string
        

In [7]:
# Load your training processed_processed_processed_processed_text here, we alse provide some sample text for you
with open('tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    train_text = f.read()

tokenizer = BPETokenizer()
# We will create a small vocabulary for demonstration purposes
tokenizer.train(train_text, vocab_size=512)

# Verification step
test_string = "O Romeo, Romeo, wherefore art thou Romeo?"
encoded_tokens = tokenizer.encode(test_string)
decoded_string = tokenizer.decode(encoded_tokens)

assert decoded_string == test_string
print("Verification Successful!")
print(f"Original string: {test_string}")
print(f"Encoded tokens: {encoded_tokens}")
print(f"Decoded string: {decoded_string}")

Verification Successful!
Original string: O Romeo, Romeo, wherefore art thou Romeo?
Encoded tokens: [462, 30, 478, 267, 30, 478, 267, 199, 419, 427, 81, 68, 196, 30, 478, 53, 116]
Decoded string: O Romeo, Romeo, wherefore art thou Romeo?


### 2.2 Question [5 pts]

**Question:** List the first 5 merge rules your tokenizer learned during training. Based on the `tinyshakespeare.txt` corpus, provide a brief explanation for why these specific pairs of characters were likely the first to be merged.

In [8]:
cnt = 0
for key, (rank, token) in tokenizer.merges.items():
    if '</w>' in key:   # ignore merges involving end-of-word token
        continue
    print(key, '->', token)
    cnt += 1
    if cnt >= 5:
        break

('t', 'h') -> th
('o', 'u') -> ou
('e', 'r') -> er
('i', 'n') -> in
('a', 'n') -> an


**Answer:**

*Your answer here. You should list the merges (e.g., ('t', 'h') -> 'th') and explain their high frequency in the context of Shakespearean English.*

The first five merge rules are: 

  `('t', 'h') -> th`

  `('o', 'u') -> ou`

  `('e', 'r') -> er`

  `('i', 'n') -> in`

  `('a', 'n') -> an`
  
These pairs were the first to be merged because the BPE Tokenizer iteratively combines the most frequent adjacent pair of characters in the corpus. These pairs are the most common pairs in Shakespearean English, forming the building blocks of high-frequency words such as "**th**e," "**th**ou," and "wi**th**" (th), "you" and "our" (ou), "nev**er**" (er), "**in**" (in), and "**an**d" (an).

## Section 3: Text generation

In this section, you will implement an LSTM-based model to generate sentences that mimic the style of the Shakespearean corpus. You will use the GloVe embeddings from Section 1 as a pre-trained embedding layer.

In [9]:
# Load necessary packages. Feel free to add ones that you need.
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

### 3.1 Load and preprocess text [5 pts]
You can choose a training corpus from the provided texts. Though the texts are much cleaner than random web crawls, you may still want to perform some preprocessing.

In [10]:
import re
# Load the text file
with open('tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

def preprocess_text(text):
    text = text.lower()
    
    # remove speaker labels
    text = re.sub(r'^[a-z \d]+:$', '', text, flags=re.MULTILINE)
    
    # remove special characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # split into paragraphs
    paragraphs = text.split('\n\n')
    processed_text = []
    processed_paragraphs = []
    
    for para in paragraphs:
        para = para.strip()
        if para:
            words = para.split()
            processed_paragraphs.append(words)
            processed_text.extend(words)
            
    return processed_text, processed_paragraphs


processed_text, processed_paragraphs = preprocess_text(text)
import random
random.seed(1116)
random.shuffle(processed_paragraphs)  # shuffle the paragraphs to ensure i.i.d. for train and val splits

In [11]:
processed_paragraphs[2]

['his',
 'majesty',
 'tendering',
 'my',
 'persons',
 'safety',
 'hath',
 'appointed',
 'this',
 'conduct',
 'to',
 'convey',
 'me',
 'to',
 'the',
 'tower']

### 3.2 Build vocabulary and setup embedding matrix [5 pts]
Create a vocabulary from your processed text. Then, create an embedding matrix where the i-th row corresponds to the GloVe vector for the i-th word in your vocabulary.

In [12]:
from collections import Counter
# Build Vocabulary from the processed text
# TODO: Create a set of unique words, then create word-to-index (word2idx) and index-to-word (idx2word) mappings.
MIN_FREQ = 2
word_counts = Counter(processed_text)
unique_words = [word for word, count in word_counts.items() if count >= MIN_FREQ]
word2idx = {'<UNK>': 0} 
idx = 1
for word in sorted(unique_words):
    word2idx[word] = idx
    idx += 1

idx2word = {idx: word for word, idx in word2idx.items()}
UNK_IDX = 0
paragraph_word_idx = []
for paragraph in processed_paragraphs:
    indices = [word2idx.get(word, UNK_IDX) for word in paragraph]
    paragraph_word_idx.append(indices)

# Create the embedding matrix from GloVe
def create_embedding_matrix(word2idx, glove_embeddings, embedding_dim):
    # Initialize matrix with zeros
    embedding_matrix = np.zeros((len(word2idx), embedding_dim))
    # TODO: 
    # For each word in your vocabulary, if it exists in glove_embeddings, 
    # add its vector to the matrix at the correct index.
    # Words not found in GloVe will remain as zero vectors.
    for word, idx in word2idx.items():
        if word in glove_embeddings:
            embedding_matrix[idx] = glove_embeddings[word]
    return torch.FloatTensor(embedding_matrix)

### 3.3 Implement the dataset [10 pts]
The text generation task uses next-word prediction as its objective. You should construct your dataset using a sliding window approach. For a sequence of length `n`, the first `n-1` words will be the input, and the `n`-th word will be the target.

In [13]:
# Construct your dataset
class TextDataset(Dataset):
    def __init__(self, paragraphs, seq_length):
        # TODO: Create input sequences and their corresponding targets.
        # For a given seq_length, each sample should be (sequence_of_indices, next_word_index).
        self.samples = []
        for para in paragraphs:
            if len(para) <= seq_length:
                continue
            for i in range(len(para) - seq_length):
                input_seq = torch.tensor(para[i:i + seq_length], dtype=torch.long)
                target = torch.tensor(para[i + 1: i + 1 + seq_length], dtype=torch.long)
                self.samples.append((input_seq, target))

    def __len__(self):
        # TODO
        return len(self.samples)

    def __getitem__(self, idx):
        # TODO
        return self.samples[idx]

### 3.4 Implement the LSTM model [10 pts]
You will use `nn.Embedding`, `nn.LSTM`, and `nn.Linear` to build your model. The embedding layer should be initialized with the pre-trained GloVe matrix.

In [14]:
# Construct your model
class TextGenLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, embedding_matrix, dropout_prob=0.5):
        super(TextGenLSTM, self).__init__()
        # TODO:  
        # 1. Create an embedding layer (nn.Embedding). Load the pre-trained embedding_matrix and set freeze=True to prevent it from being trained.
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)
        # 2. Create an LSTM layer (nn.LSTM).
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        # 3. Create a fully connected layer (nn.Linear) to map LSTM output to vocabulary size.
        self.dropout = nn.Dropout(p=dropout_prob)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        # TODO
        embeds = self.embedding(x)
        lstm_out, hidden = self.lstm(embeds, hidden)
        out = self.dropout(lstm_out)
        out = self.fc(out)
        return out, hidden

### 3.5 Implement a generate_text function [10 pts]
This function will take a starting sequence (prompt) and generate a specified number of new words. To get more interesting results than simple greedy decoding (always picking the most probable word), try implementing a sampling strategy like top-k sampling.

In [15]:
# Generate text with your model
def generate_text(model, start_sequence, num_words_to_generate, word2idx, idx2word, device, top_k=5):
    """
    Generates text using the trained model and a top-k sampling strategy.
    """
    # TODO:
    # 1. Set the model to evaluation mode.
    model.eval()
    # 2. Convert start_sequence to a tensor of indices.
    input_indices = [word2idx.get(word) for word in start_sequence.split()]
    input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)
    hidden = None
    generated_indices = []
    # 3. Generate one word at a time for num_words_to_generate:
    with torch.no_grad():
        UNK_IDX = word2idx.get('<UNK>', 0)
        for _ in range(num_words_to_generate):
            # a. Feed the current sequence to the model.
            # b. Get the output logits for the next word.
            outputs, hidden = model(input_tensor, hidden)
            # c. Apply top-k sampling: get the top k logits and their indices, 
            #    convert them to probabilities using softmax, and sample from this new distribution.
            logits = outputs[:, -1, :]
            logits[:, UNK_IDX] = -torch.inf  # prevent sampling <UNK>
            top_k_logits, top_k_indices = torch.topk(logits, top_k)
            
            probabilities = torch.softmax(top_k_logits, dim=-1)
            sampled_index = torch.multinomial(probabilities, 1).item()
            sampled_word_index = top_k_indices[0][sampled_index].item() # type: ignore
            # d. Append the sampled word's index to the sequence and use it as input for the next step.
            input_tensor = torch.tensor([[sampled_word_index]], dtype=torch.long).to(device)
            generated_indices.append(sampled_word_index)
        # 4. Convert the final sequence of indices back to words and return as a string.
        generated_words = [idx2word[idx] for idx in generated_indices]
    return ' '.join(generated_words)

### 3.6 Implement the training loop [10 pts]
Train your model. During each epoch, log the average training and validation loss. It's also highly recommended to generate a short piece of text after each epoch to see how the model's creative abilities evolve.

In [16]:
import tqdm

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            inputs, targets = inputs.to(device), targets.to(device)
            # TODO: Your training steps here (zero grad, forward pass, loss, backward, step)
            # Remember to detach the hidden state to prevent backpropagating through the entire history.
            optimizer.zero_grad()
            outputs, hidden = model(inputs, None)
            # print(outputs.shape, targets.shape)
            loss = criterion(outputs.view(-1, outputs.shape[2]), targets.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        # Calculate average training loss
        avg_train_loss = train_loss / len(train_loader)

        # Validation loop
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                # TODO: Your validation steps here
                outputs, hidden = model(inputs, None)
                loss = criterion(outputs.view(-1, outputs.shape[2]), targets.view(-1))
                val_loss += loss.item()
        
        scheduler.step()

        # Calculate average validation loss
        avg_val_loss = val_loss / len(val_loader)

        # Logging and generating sample text
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
        
        # TODO: Call your generate_text function with a fixed prompt (e.g., "shall i compare thee")
        # and print the generated text to observe the model's progress.
        prompt = "shall i compare thee"
        generated_text = generate_text(model, prompt, num_words_to_generate=10, 
                                       word2idx=word2idx, idx2word=idx2word, 
                                       device=device, top_k=10)
        print(f"  [Sample]: {prompt} ... {generated_text}\n")
    return model

In [19]:
# Initialize hyperparameters, model, optimizer, etc., and start the training process.
EMBEDDING_DIM = 100 # GloVe embedding dimension is 100
VOCAB_SIZE = len(word2idx)
HIDDEN_DIM = 256
NUM_LAYERS = 1
SEQ_LENGTH = 32
BATCH_SIZE = 128
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 50
VAL_SPLIT = 0.1
DROPOUT_PROB = 0.5

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

print("Loading and splitting data...")
# split corpus first to avoid data leakage
# note that paragraph_word_idx is already shuffled
split_idx = int(len(paragraph_word_idx) * (1 - VAL_SPLIT))
train_indices = paragraph_word_idx[:split_idx]
val_indices = paragraph_word_idx[split_idx:]
print(f"  Total paragraphs: {len(paragraph_word_idx)}")
print(f"  Training paragraphs: {len(train_indices)}")
print(f"  Validation paragraphs: {len(val_indices)}")
train_dataset = TextDataset(train_indices, SEQ_LENGTH)
val_dataset = TextDataset(val_indices, SEQ_LENGTH)
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Initializing model...")
glove_embeddings = load_glove_embeddings(glove_file_path)
embedding_matrix = create_embedding_matrix(word2idx, glove_embeddings, EMBEDDING_DIM)
model = TextGenLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    embedding_matrix=embedding_matrix,
    dropout_prob=DROPOUT_PROB
).to(device)
criterion = nn.CrossEntropyLoss()

embedding_params = model.embedding.parameters()
other_params = [param for name, param in model.named_parameters() if 'embedding' not in name]

optimizer = torch.optim.Adam(
    [
        {'params': embedding_params, 'lr': 1e-5},
        {'params': other_params, 'lr': LEARNING_RATE},
    ], 
    lr=1e-3,
    weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Starting training process...")
trained_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS
)

Using device: mps
Loading and splitting data...
  Total paragraphs: 7564
  Training paragraphs: 6807
  Validation paragraphs: 757
  Training samples: 58882
  Validation samples: 4729
Initializing model...
Starting training process...


Epoch 1/50: 100%|██████████| 460/460 [00:15<00:00, 29.51it/s]


Epoch 1/50, Train Loss: 6.6334, Val Loss: 6.4147
  [Sample]: shall i compare thee ... to the that of and i have not have the



Epoch 2/50: 100%|██████████| 460/460 [00:15<00:00, 29.78it/s]


Epoch 2/50, Train Loss: 6.2529, Val Loss: 6.2524
  [Sample]: shall i compare thee ... to me in this and and a his that i



...

Epoch 48/50, Train Loss: 5.2512, Val Loss: 5.9007
  [Sample]: shall i compare thee ... for me i am the rest of thee thou hast



Epoch 49/50: 100%|██████████| 460/460 [00:15<00:00, 30.41it/s]


Epoch 49/50, Train Loss: 5.2507, Val Loss: 5.9007
  [Sample]: shall i compare thee ... to my heart that that thou dost me to my



Epoch 50/50: 100%|██████████| 460/460 [00:15<00:00, 29.64it/s]


Epoch 50/50, Train Loss: 5.2512, Val Loss: 5.9007
  [Sample]: shall i compare thee ... i am not in a king to my love my



### 3.7 Question: Design Choices [15 pts]

**Question:** Discuss **at least two** design choices you made during the implementation of your text generation model (Section 3) and explain how they impacted the final result. You can discuss any of the steps, from text preprocessing and dataset construction to model architecture and the text generation strategy.

For each choice, describe:
1.  **What was the choice?** (e.g., sequence length in the dataset, number of LSTM layers, using top-k sampling vs. greedy decoding).
2.  **What was your rationale for this choice?** (e.g., 'I chose a longer sequence length to capture more context...' or 'I used top-k sampling to avoid repetitive text...').
3.  **How did it affect the outcome?** (e.g., 'This resulted in more coherent sentences but increased training time.' or 'The generated text became more diverse and less predictable.').

*Your answer will be evaluated based on the clarity and depth of your rationale. **Please note:** The goal of this question is to encourage reflection. As long as you clearly explain your choices and your reasoning, you will receive full credit, so you don't need to write a lot and worry about losing points.*

Answer: 

### Design Choice 1: Data Preprocessing
I choose to replace words that showed up less frequently than a certain threshold with an `<UNK>` token. 

This is done to reduce the vocabulary size and help the model focus on learning the patterns of more common words. 

By limiting the vocabulary, the model can generalize better and avoid overfitting to rare words. It also helped in speeding up the training process since the fc layer has fewer parameters to learn.

### Design Choice 2: Many-to-Many Prediction
I change the model to a many-to-many prediction, where the model predicts the next word at each time step instead of just the last word in the sequence.

This allows the model to learn from more gradient signals per input sequence, which can lead to better learning of temporal dependencies in the text.

By providing feedback at each time step, the model can adjust its weights more frequently, leading to improved performance and more coherent text generation, but also increases the computational time during training.

### Design Choice 3: Unfrozen Embeddings
Since I reduce the vocabulary size by replacing rare words with `<UNK>`, which leads to a smaller embedding matrix. Thus I choose to keep the embedding layer unfrozen during training.

This allows the model to fine-tune the embeddings to better fit the specific patterns and vocabulary of the Shakespearean text, improving overall performance. 

Note that when the vocabulary is large, unfreezing the embeddings may lead to overfitting, so it's necessary to reduce the vocabulary size first.